# 05Gb: Whole-model evaluation (Phase G2b, exploratory)

Evaluates the whole-model explanations from **04Ge** along the two axes the per-feature
track (05G) cannot address. Everything reuses the G3 scorers: the split records
(`results/global_whole_split/`) are byte-compatible with the G2a per-feature records, so
the same deterministic rubric (`utils.rubric`) and reference-based judge
(`utils.eval.run_global_judge`, `src_subdir="global_whole_split"`) apply unchanged.

- **Coverage**: how many of the 9 features each whole-model answer actually described
  (the meeting's "only 5 of 9 right" concern, made measurable).
- **Axis 1, representation**: all 9 curves/plots vs the single beeswarm, reported
  **per GT field** (no aggregate winner); the beeswarm is scored only on the fields it
  can carry (direction, rank) via `fair_total`.
- **Axis 2, mechanism**: full-push (`json_all`/`vision_all`) vs pull (`tooluse_all`) at
  constant full information.
- **Beeswarm readability**: `vision_beeswarm` vs the info-matched `json_beeswarm`:
  same information, different modality → the pure "can the LLM *read* the swarm?" effect.

> Runs on the **real** split records if 04Ge was run with `RUN_API=True`; otherwise it
> falls back to a **STUB** (deterministic, correct ranks/direction) so every table and
> code path is verified without API cost. The reference judge is separately guarded
> (`RUN_JUDGE`).

In [1]:
from __future__ import annotations

import sys, json, tempfile
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from utils import (
    RESULTS_DIR, EXPLANATIONS_DIR, WHOLE_SPLIT_SUBDIR, WHOLE_CONDITIONS,
    list_global_features, feature_importance_map, describe_curve,
    build_whole_record, write_split_records,
)
from utils import global_eval

XAI_MODELS = ['xgb', 'ebm']
FEATURES   = list_global_features('ebm', explanations_dir=EXPLANATIONS_DIR)

df = global_eval.load_whole_rubric()      # real split records, or None if 04Ge not run
STUB = df is None

if STUB:
    def _stub_expl(model_name):
        imp = feature_importance_map(model_name, explanations_dir=EXPLANATIONS_DIR)
        lines = ['<analysis>stub</analysis>']
        for f in FEATURES:
            d = describe_curve(model_name, f, explanations_dir=EXPLANATIONS_DIR)
            lines += [f'[FEATURE: {f}]',
                      f'[EFFECT] The effect is {d["direction"]} and {d["monotonicity"]}.',
                      f'[IMPORTANCE] Rank {imp[f]["rank"]} of {len(FEATURES)}.']
        lines.append('[RECOMMENDATION] Plan bikes and staff around the top drivers.')
        return '\n'.join(lines)

    recs = []
    for m in XAI_MODELS:
        for c in WHOLE_CONDITIONS:
            txt = _stub_expl(m)
            if c.name == 'vision_beeswarm':               # drop last feature -> coverage < 9
                txt = txt.rsplit(f'[FEATURE: {FEATURES[-1]}]', 1)[0].rstrip() + \
                      '\n[RECOMMENDATION] Plan around the top drivers.'
            recs.append(build_whole_record(condition=c, model_name=m, explanation=txt,
                                           usage={}, llm_model='stub'))
    tmp = Path(tempfile.mkdtemp())
    write_split_records(recs, features=FEATURES, split_dir=tmp / WHOLE_SPLIT_SUBDIR)
    df = global_eval.load_whole_rubric(results_dir=tmp)
    print(f'No real split records found -> STUB verification on {len(df)} rows.')
    print('(Values are placeholders; only the labels/shapes are meaningful until 04Ge runs.)')
else:
    print(f'Loaded {len(df)} real whole-model split records.')

df[['condition', 'xai_model', 'feature', 'form_type', 'direction', 'rank',
    'structure', 'total', 'fair_total', 'dropped']].head(6)

Loaded 90 real whole-model split records.


,condition,xai_model,feature,form_type,direction,rank,structure,total,fair_total,dropped
0,json_all,ebm,holiday,near-flat,1.00,0.0,1.0,0.6667,0.6667,False
1,json_all,ebm,hr,categorical,1.00,1.0,1.0,1.0000,1.0000,False
2,json_all,ebm,hum,non-monotonic,0.75,0.0,0.5,0.4167,0.4167,False
3,json_all,ebm,mnth,categorical,1.00,0.0,1.0,0.6667,0.6667,False
4,json_all,ebm,temp,non-monotonic,1.00,1.0,1.0,1.0000,1.0000,False
5,json_all,ebm,weathersit,categorical,1.00,0.0,1.0,0.6667,0.6667,False


## 1. Coverage: how many of the 9 features were described?

A whole-model answer that silently drops features is the core scoring risk from the
meeting. `dropped=True` feature blocks are scored as total misses.

In [2]:
cov = global_eval.whole_coverage(df)
print('Features described (of 9), by condition x model:')
print(cov)

Features described (of 9), by condition x model:
xai_model        ebm  xgb
condition                
json_all           9    9
vision_all         9    9
tooluse_all        9    9
json_beeswarm      9    9
vision_beeswarm    9    9


## 2. Axis 1: representation (all vs beeswarm), per GT field

The two representations carry **different** information, so there is no single winner:
report each GT field. Expectation a priori: all-plots/curves win on `structure`
(shape/peak), the beeswarm stays competitive on `direction`/`rank`. Read the beeswarm
rows on `direction`/`rank` only; `fair_total` already restricts the beeswarm to those.

In [3]:
a1 = global_eval.axis1_representation(df)
print('Mean rubric sub-scores per condition (beeswarm: read direction/rank; fair_total is field-fair):')
print(a1)

Mean rubric sub-scores per condition (beeswarm: read direction/rank; fair_total is field-fair):
                 direction   rank  structure  fair_total
condition                                               
json_all             0.986  0.611      0.917       0.838
vision_all           0.944  0.111      0.750       0.602
tooluse_all          0.986  0.889      0.917       0.931
json_beeswarm        0.972  1.000      0.556       0.986
vision_beeswarm      0.889  0.944      0.667       0.917


## 3. Axis 2: mechanism (push vs pull) at full information

Restricted to the `all` conditions (constant full information): full-push
(`vision_all`/`json_all`, mechanism = push) vs pull (`tooluse_all`).

Pooling both push conditions mixes **mechanism** with **modality** (`limitations.md` 4.4):
if one push condition is weak, the pooled mean makes pull look uniformly better. So the
pooled view is reported **together with** the per-condition breakdown — a negative delta
means that push condition beat pull.

In [ ]:
a2 = global_eval.axis2_mechanism(df, value='total')
print('Pooled - mean rubric total by mechanism x model (all-information conditions only):')
print(a2)

print('\nDisaggregated - pull vs each push condition separately (fair_total):')
print(global_eval.axis2_mechanism_pairwise(df, value='fair_total').to_string(index=False))
print('\nA pooled push mean that rests on one weak condition shows up here as a large '
      'spread between the two push rows.')

## 4. Beeswarm readability: image vs info-matched numbers

`vision_beeswarm` (the swarm image) vs `json_beeswarm` (the same ranking + colour
direction + spread as numbers). Same information, so the gap is the **pure modality
effect**: can the LLM read the swarm as well as it reads the equivalent numbers?

In [5]:
bee = global_eval.beeswarm_readability(df, value='fair_total')
print('Mean fair_total (direction+rank) — swarm image vs info-matched numbers:')
print(bee)

Mean fair_total (direction+rank) — swarm image vs info-matched numbers:
xai_model          ebm    xgb
condition                    
json_beeswarm    1.000  0.972
vision_beeswarm  0.917  0.917


## 5. Reference-based judge (billed, guarded)

Scores the whole-model split records with the same reference judge as 05G, via
`src_subdir="global_whole_split"`. Runs **both vendors** (Anthropic + OpenAI/gpt-4o) so the
whole-model track gets the same **cross-vendor Krippendorff-alpha** robustness figure as
the G2a per-feature track. Idempotent; the frozen G2a judge outputs are untouched.

In [6]:
RUN_JUDGE = True   # <- billed: scores the whole-model split records (Anthropic + OpenAI)

if STUB:
    print('STUB mode: skipping the billed judge (run 04Ge with RUN_API=True first).')
elif RUN_JUDGE:
    from utils.eval import run_global_judge
    from utils.llm import ask_text, ask_openai_text, OPENAI_JUDGE_MODEL_FINAL
    ANTHROPIC_JUDGE = 'claude-opus-4-8'
    OPENAI_JUDGE    = OPENAI_JUDGE_MODEL_FINAL   # gpt-4o, cross-vendor

    # primary judge (Anthropic) - idempotent, skips the records already scored
    aj = run_global_judge(ask_text, ANTHROPIC_JUDGE,
                          src_subdir='global_whole_split', out_subdir='global_whole_judge')

    # cross-vendor judge (OpenAI), same reference-based rubric + split records
    def _ask_openai(prompt, *, system, model, max_tokens, cache_system=None, temperature=None):
        return ask_openai_text(prompt, system=system, model=model,
                               max_tokens=max_tokens, temperature=temperature)
    oj = run_global_judge(_ask_openai, OPENAI_JUDGE,
                          src_subdir='global_whole_split', out_subdir='global_whole_judge_openai')
    print(f'Judged {len(aj)} (Anthropic) + {len(oj)} (OpenAI) whole-model split records.')

    print('\nAnthropic - mean faithfulness/clarity/completeness by condition:')
    display(aj.groupby('form_pipeline')[['faithfulness','clarity','completeness']].mean().round(2))
    print('OpenAI - mean faithfulness/clarity/completeness by condition:')
    display(oj.groupby('form_pipeline')[['faithfulness','clarity','completeness']].mean().round(2))

    # cross-vendor robustness (Krippendorff alpha, interval) on the whole-model track,
    # the same estimator used for the G2a per-feature judge in 05G
    vendors = ['global_whole_judge', 'global_whole_judge_openai']
    print('\nCross-vendor Krippendorff alpha (whole-model, 90 units):')
    for crit in global_eval.JUDGE_CRITERIA:
        print(f'  {crit:13s}: {global_eval.cross_vendor_alpha(vendors, crit):.3f}')
else:
    print('RUN_JUDGE=False -> skipped. Set True to score the whole-model split records.')

Judged 90 (Anthropic) + 90 (OpenAI) whole-model split records.

Anthropic - mean faithfulness/clarity/completeness by condition:


,faithfulness,clarity,completeness
form_pipeline,,,
json_all,4.72,4.33,3.61
json_beeswarm,3.61,4.17,3.72
tooluse_all,4.89,4.11,3.89
vision_all,4.22,3.28,2.94
vision_beeswarm,3.89,3.39,4.00


OpenAI - mean faithfulness/clarity/completeness by condition:


,faithfulness,clarity,completeness
form_pipeline,,,
json_all,3.39,4.56,4.33
json_beeswarm,3.44,4.67,3.78
tooluse_all,3.72,4.56,4.28
vision_all,3.06,3.78,3.39
vision_beeswarm,2.72,3.67,3.94



Cross-vendor Krippendorff alpha (whole-model, 90 units):
  faithfulness : 0.329
  clarity      : 0.584
  completeness : 0.623
